# Beginner Machine Learning Practice — SCM Stock Status

**Audience:** Beginner with little or no machine-learning knowledge  
**Environment:** Databricks notebook using Python, pandas, and scikit-learn  
**Time:** Work at your own pace; run one cell at a time  
**Purpose:** Understand the basic machine-learning workflow through a small, understandable inventory example.

## Business problem

Imagine a computer manufacturer wants to classify an inventory item as `Low` or `Normal` based on its available quantity. We will create a tiny synthetic dataset, teach a model from labelled examples, ask it to predict labels for new quantities, and measure its performance.

This is a learning exercise, not a real inventory policy. The sample labels are invented for practice, and quantity alone is not enough for a production stock-risk decision. A real system may need demand, supplier lead time, open orders, safety stock, seasonality, and other information.

## Learning outcomes

By the end, you should be able to:
1. Explain what machine learning does in this example.
2. Distinguish a dataset, a feature (`X`), and a target (`y`).
3. Explain why data is explored before training.
4. Explain training/test split, `fit()`, and `predict()`.
5. Read accuracy and a confusion matrix at a basic level.
6. Change the data and observe how the model's results can change.
7. Identify what this notebook does **not** cover yet (MLflow, model registry, deployment).

## Roadmap

1. Create labelled examples
2. Explore the data (EDA)
3. Separate feature and target
4. Split data into training and test sets
5. Train a decision-tree classifier
6. Predict new examples
7. Evaluate predictions
8. Practice changing data and model settings
9. Review key terms and connect this exercise to the course

**Please do not rush.** Your goal is to understand each step, not to finish a course or memorize every line.

## Before coding: what is machine learning?

In traditional programming, a developer writes explicit rules. For example, a rule might say: “if quantity is below 15, label it Low.” In machine learning, we give an algorithm examples with known answers, and it learns a pattern from those examples.

In this notebook:
- **Examples/rows:** individual item observations.
- **Feature/input:** `available_quantity`.
- **Label/target:** `stock_status`, the known answer.
- **Model:** the trained object that learns a relationship between input and label.
- **Prediction:** the model's answer for an input it has not been given a label for in that prediction step.
- **Evaluation:** comparing predictions against known correct answers on held-out data.

### Classification vs. regression
- **Classification** predicts a category, such as `Low` or `Normal`.
- **Regression** predicts a numeric value, such as expected demand next week.

We use classification here because `stock_status` is a category.

## Cell 1 — Create sample data

Each row is one example. The `stock_status` column is the answer we provide during training.

In [0]:
import pandas as pd

data = {
    "available_quantity": [3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 60],
    "stock_status": [
        "Low", "Low", "Low", "Low", "Low", "Low",
        "Normal", "Normal", "Normal", "Normal", "Normal", "Normal"
    ]
}

df = pd.DataFrame(data)
display(df)

### What the code means
"""
- `import pandas as pd` loads the pandas library and gives it the short name `pd`.
- `data` is a Python dictionary: each key becomes a column name, and each list supplies that column's values.
- `pd.DataFrame(data)` turns the dictionary into a table.
- `display(df)` shows the table in Databricks.
"""

### Expected outcome
"""
You should see 12 rows and 2 columns. Six rows are labelled `Low`, and six are labelled `Normal`. The rows are synthetic examples created for this lesson.
"""

**Understand the outcome:** `df` is our labelled example data. Real ML training needs examples where the correct answer is known. Here, we created the answers ourselves so we can practice safely.

## Cell 2 — Explore the data (EDA)

EDA means looking at the data before training. Check row counts, data types, label distribution, and numeric summaries.

### How to interpret the results
- `df.shape` returns `(rows, columns)`.
- `df.info()` shows column names, data types, and non-null counts.
- `value_counts()` counts how many rows belong to each stock-status category.
- `describe()` gives numeric summaries such as count, mean, standard deviation, minimum, quartiles, and maximum.

### Why EDA matters
EDA can reveal missing values, unexpected data types, extreme values, duplicated rows, and imbalanced labels before training. A model cannot automatically know whether the input data is correct.

In [0]:
print("Rows and columns:", df.shape)
print("\nData types and non-null values:")
df.info()

print("\nExamples per status:")
print(df["stock_status"].value_counts())

print("\nQuantity summary:")
display(df["available_quantity"].describe().to_frame())

**Questions to answer:**
- How many rows are in this dataset?
- How many are labelled `Low`?
- What is the minimum and maximum quantity?
- Why might a model struggle if the examples were missing or incorrectly labelled?

### Why the syntax is different
- `df[["available_quantity"]]` uses double brackets and returns a two-dimensional DataFrame, which is the expected shape for model features.
- `df["stock_status"]` returns a one-dimensional Series for the target.
- `X` is often used as the conventional name for features; `y` is commonly used for the target.

The model learns a relationship between `X` and `y`; it should not receive the target as an input feature when predicting.

## Cell 3 — Separate features and target

`X` is the information the model is allowed to use. `y` is the answer we want it to learn to predict.

In [0]:
X = df[["available_quantity"]]
y = df["stock_status"]

print("Features (X):")
display(X.head())

print("Target (y):")
display(y.head())

### Why not train and test on the same rows?
"""
If we evaluate a model using only the examples it learned from, the score may look too optimistic. A test set gives us a basic check on examples held aside from training.

- `X_train`, `y_train`: examples used to learn.
- `X_test`, `y_test`: examples held aside for evaluation.
- `stratify=y`: tries to preserve the label proportions in both sets.
- `random_state=42`: makes the split reproducible in repeated runs with the same data and environment.

With only 12 rows, the test set is extremely small, so the score is only a teaching illustration.
"""

**Key idea:** We keep the feature as a DataFrame (`df[[...]]`) because scikit-learn expects a two-dimensional input matrix. The target is a Series (`df[...]`).

## Cell 4 — Split training and test examples

Training examples are used to fit the model. Test examples are held aside to estimate how it performs on examples it did not train on.

### What happens during `fit()`?
`DecisionTreeClassifier` is an algorithm for predicting categories. A decision tree learns split rules from the training examples. `max_depth=2` limits the tree's depth so the model stays relatively simple for this first lesson.

- `DecisionTreeClassifier(...)` creates the model object and configures it.
- `model.fit(X_train, y_train)` learns from the training inputs and known labels.
- `random_state=42` makes any randomness reproducible.

The model is not a human and does not understand inventory in a business sense. It learns statistical patterns from the examples supplied.

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTest labels:")
print(y_test.value_counts())

`test_size=0.25` reserves about one quarter of the examples for testing. `random_state=42` makes the split repeatable. `stratify=y` tries to preserve the proportions of the labels in each split.

This dataset is tiny, so the evaluation score can change noticeably with different splits.

### What happens during `predict()`?
`model.predict(new_items)` applies the learned pattern to each row in `new_items` and returns a predicted category. We add those predictions to a copy of the input DataFrame so the input and output can be read side by side.

The model is not being told the correct status for these new quantities in this cell. It estimates a label based on its training.

## Cell 5 — Train a simple model

A decision tree learns rules by splitting the data. We limit its depth to keep this first example simple.

In [0]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(X_train, y_train)

print("Training complete.")

### Metrics in simple language
"""
- **Accuracy:** number of correct test predictions divided by total test predictions.
- **Confusion matrix:** a table comparing actual labels with predicted labels. In the code, the order is explicitly `Low`, then `Normal`.
- **Precision:** when the model predicts a class, how often that prediction is correct.
- **Recall:** of all actual examples in a class, how many the model identifies.
- **F1-score:** a combined measure of precision and recall.

For stock-risk use cases, a false `Normal` prediction for an item that is actually at risk could be more costly than a false `Low` prediction. That is why accuracy alone may not be enough in a real system.

The dataset is deliberately small and synthetic. Do not use its score to judge a production model.
"""

**What `fit()` does:** it uses the training features and their known labels to learn a pattern. We are not manually writing an `if quantity < ...` rule; the algorithm chooses splits from the examples.

## Cell 6 — Predict for new quantities

Now we give the trained model quantities for which we have not supplied a status in this cell.

In [0]:
new_items = pd.DataFrame({"available_quantity": [6, 18, 35]})
predictions = model.predict(new_items)

results = new_items.copy()
results["predicted_stock_status"] = predictions
display(results)

**Outcome:** each new quantity gets a predicted category. The model is making a learned prediction, not looking up a label from `df`.

Try changing the quantities to `[2, 22, 45]` and run this cell again.

## Cell 7 — Evaluate on held-out test data

Accuracy is the fraction of test predictions that are correct. A confusion matrix shows correct and incorrect predictions by class.

In [0]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

test_predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("\nConfusion matrix (class order: Low, Normal):")
print(confusion_matrix(y_test, test_predictions, labels=["Low", "Normal"]))

print("\nClassification report:")
print(classification_report(y_test, test_predictions, zero_division=0))

**How to interpret this:**
- Accuracy ranges from 0 to 1; for example, `0.75` means 75% of this test split was predicted correctly.
- The confusion matrix helps reveal which class the model gets wrong.
- Do not treat a high score on this tiny, artificial dataset as evidence that it will work well in production.

## Cell 8 — Practice changing the examples

Change one or more quantities or labels in Cell 1, then rerun the notebook from Cell 1 downward. Observe how the model and its evaluation change.

Suggested experiments:
1. Change the quantity `15` to `35` but leave its label as `Low`. What does this tell you about data quality?
2. Add examples from a wider range of quantities.
3. Remove some examples and compare results.
4. Change `max_depth=2` to `max_depth=1` and compare predictions. A shallower tree is simpler, but may miss patterns.

**Important:** If you change the dataset, rerun all cells in order so `X`, `y`, the split, and the model all use the updated data.

## Final recap

After this practice, you should be able to explain:

- **Dataset:** examples used for learning.
- **Feature (`X`):** input information.
- **Target (`y`):** answer to predict.
- **Training (`fit`):** learning patterns from labelled examples.
- **Prediction (`predict`):** applying the learned model to new inputs.
- **Evaluation:** checking predictions against known test answers.

### How this connects to the Academy material

This notebook focuses on the foundations needed before the course's later steps:
- EDA and feature preparation
- Model training and evaluation
- MLflow tracking and model management (later)
- Mosaic AI Model Serving and workflows (later)

Stop here for now. Your goal is understanding the flow, not completing every topic in one sitting.

## Detailed reference: what each stage achieves

| Stage | What we do | Why we do it | Output |
|---|---|---|---|
| Create data | Build a small table with quantities and known labels | A supervised model needs examples with answers | `df` |
| EDA | Inspect shape, types, labels, and summary statistics | Catch issues and understand the data | A basic understanding of the dataset |
| Feature/target split | Put input columns in `X` and answer column in `y` | Tell the algorithm what it may use and what it should learn to predict | `X`, `y` |
| Train/test split | Reserve some rows for testing | Estimate performance on held-out examples | `X_train`, `X_test`, `y_train`, `y_test` |
| Train | Call `model.fit(...)` | Learn a pattern from examples | A fitted model |
| Predict | Call `model.predict(...)` | Produce predictions for new inputs | Predicted categories |
| Evaluate | Compare predictions with known test labels | Understand errors and performance | Accuracy and classification report |

## Expected outcome of this notebook

When you finish, you will have:
- A small labelled dataset.
- A trained decision-tree classification model.
- Predictions for three new stock quantities.
- Basic evaluation metrics on a held-out test set.
- Hands-on experience changing data and model settings.

You will **not** have:
- A validated inventory forecasting system.
- A model trained on your one-million-item SCM table.
- MLflow experiment tracking.
- A registered model or serving endpoint.
- A production-ready recommendation for inventory replenishment.

Those are later learning steps.

## Common beginner questions

### Does the model know the stock status automatically?
No. During training, we provide examples with known labels. The model learns a pattern from those examples.

### Is `Low` a universal stock threshold?
No. The labels in this notebook are made up for practice. Real thresholds depend on demand, lead time, safety stock, service-level targets, and business rules.

### Why do we need a test set?
To check predictions on examples that were not used to fit the model. It provides a more useful first estimate than testing only on training data.

### Why can the accuracy change?
The dataset is tiny. Changing the split, examples, or tree depth can change the learned model and its score. On real tasks, use enough representative data and an evaluation design appropriate to the problem.

### Why use a decision tree?
It is a simple model for learning category labels and is often easier to explain to beginners than more complex models.

### Does a high accuracy mean the model is good?
Not necessarily. It depends on the dataset, label quality, class balance, error costs, and whether test data represents future cases. In real work, compare multiple relevant metrics and validate on representative data.

### Can I use this directly on my million-row Databricks dataset?
Not yet. This notebook intentionally uses a tiny pandas DataFrame so the workflow is easy to understand. Scaling to a real Spark dataset requires additional data preparation, suitable evaluation, and model workflow choices.

## Practice tasks with expected learning

1. **Change one quantity but not its label.**  
   Learning: inconsistent labels can confuse the model. In real projects, data quality is critical.

2. **Add examples near the boundary between `Low` and `Normal`.**  
   Learning: examples near ambiguous cases influence what the model learns.

3. **Change `max_depth=2` to `max_depth=1`.**  
   Learning: simpler models may be easier to interpret but may miss patterns.

4. **Change the test split.**  
   Learning: small test sets can produce unstable metrics. Keep `random_state` fixed when comparing one change at a time.

5. **Explain the workflow without looking at code.**  
   Learning: understanding the process is more valuable than memorizing syntax.

After changing the original dataset, rerun the notebook from the first code cell in order. Otherwise, later variables may still refer to old data.

## Glossary

- **Dataset:** a collection of examples arranged in rows and columns.
- **Row/example:** one observation.
- **Feature:** an input variable used by a model.
- **Target/label:** the answer to be predicted.
- **Supervised learning:** learning from examples that include known answers.
- **Classification:** predicting a category.
- **Regression:** predicting a numeric value.
- **Training:** fitting a model to examples.
- **Test set:** examples held aside to evaluate the model.
- **Model:** the fitted object that maps inputs to predictions.
- **Prediction:** the model's output for new input data.
- **Metric:** a numeric summary of model performance.
- **Overfitting:** when a model fits training examples too closely and performs poorly on new data.
- **MLflow:** a tool commonly used to track experiments and manage model lifecycle. It is not used in this first notebook.
- **Model serving:** exposing a trained model for inference through a serving interface. It is not used in this first notebook.

## Connection to the Databricks Academy course

This notebook is a gentle foundation for topics such as EDA, feature preparation, model training, and evaluation. The Academy material's later MLflow, model registry, and Mosaic AI Model Serving sections should be approached after these basic concepts make sense.

**Suggested stopping point:** run the notebook, complete at least two practice tasks, and make sure you can explain `X`, `y`, `fit()`, `predict()`, and accuracy in your own words. Do not worry about completing the entire course today.